In [0]:
import pandas as pd
import numpy as np

In [0]:
from pathlib import Path

project_root = Path.cwd().parent.parent
df_cc = spark.read.table("workspace.`ds-raw-datasets`.raw_cc_calls").toPandas()

display(df_cc.head(10))

In [0]:
df_cc.dtypes

In [0]:
df_cc.shape

In [0]:
# Drop if every value in the column is missing
df_cc = df_cc.dropna(axis=1, how='all')

# Remove Unnamed columns
df_cc = df_cc.loc[:, ~df_cc.columns.str.contains('^Unnamed')]

print(f"Columns remaining: {df_cc.columns.tolist()}")

In [0]:
print(df_cc.isnull().sum())

In [0]:
for col in df_cc.columns:
    if col not in ['Co_Ref', 'Contact_ID']:
        uc = df_cc[col].nunique()
        print(f"\n--- {col} ({uc} unique) ---")
        if uc < 20:
            print(df_cc[col].value_counts(dropna=False))
        else:
            print(df_cc[col].value_counts(dropna=False).head(15))

In [0]:
print(f"Null co_ref before: {df_cc['co_ref'].isnull().sum()}")

# Build a lookup from rows that have a valid co_ref
has_coref = df_cc[df_cc['co_ref'].notna()]
contact_to_coref = has_coref.drop_duplicates(subset='contact_id').set_index('contact_id')['co_ref']

# Fill missing co_ref by looking up contact_id
null_mask = df_cc['co_ref'].isna()
df_cc.loc[null_mask, 'co_ref'] = df_cc.loc[null_mask, 'contact_id'].map(contact_to_coref)

print(f"Null co_ref after contact_id lookup: {df_cc['co_ref'].isnull().sum()}")

In [0]:
# For remaining nulls, fill with UNKNOWN_ + contact_id
still_null = df_cc['co_ref'].isna()
df_cc.loc[still_null, 'co_ref'] = 'UNKNOWN_' + df_cc.loc[still_null, 'contact_id'].astype(str)

print(f"Null co_ref after placeholder fill: {df_cc['co_ref'].isnull().sum()}")
print(f"\nPlaceholder entries:")
print(df_cc[df_cc['co_ref'].str.startswith('UNKNOWN_', na=False)][['contact_id', 'co_ref']])

In [0]:
# Care Package related columns - fill null with 'Not Discussed'
care_pkg_cols = ['cc_care_package', 'cc_care_package_discussed']
for col in care_pkg_cols:
    df_cc[col] = df_cc[col].fillna('Not Discussed')

# Call-related general columns - fill null with 'Not Discussed'
general_cols = [
    'cc_urgency_getting_on_site', 'cc_external_consultant',
    'cc_agent_cross_sell_attempt', 'cc_customer_issues_concerns',
    'cc_business_struggles_financial_hardship', 'cc_call_initiated_by',
    'cc_pricing_mentioned', 'cc_pricing_sentiment_impact',
    'cc_refund_discussed', 'cc_contractor_suggest_leave',
    'cc_contractor_complained'
]
for col in general_cols:
    df_cc[col] = df_cc[col].fillna('Not Discussed')

# Sentiment columns - fill null with 'Neutral'
sentiment_cols = ['cc_contractor_sentiment', 'cc_dissatisfaction_support']
for col in sentiment_cols:
    df_cc[col] = df_cc[col].fillna('Neutral')

# Sentiment score columns - fill null with median later during numeric conversion
# (handled in score cleaning step below)

# Technical / Questionnaire issue columns - fill null with 'No' (no issue)
tech_cols = [
    'cc_questionnaire_completion', 'cc_chasing_response',
    'cc_issues_within_questionnaire', 'cc_login_issues',
    'cc_platform_issues', 'cc_dissatisfaction_time_to_complete',
    'cc_process_complexity_concerns', 'cc_questions_harder_than_expected'
]
for col in tech_cols:
    df_cc[col] = df_cc[col].fillna('No')

print("Null counts after grouped fill:")
print(df_cc.isnull().sum())

In [0]:
flag_cols = [
    'cc_care_package_discussed', 'cc_urgency_getting_on_site', 'cc_external_consultant',
    'cc_agent_cross_sell_attempt', 'cc_customer_issues_concerns',
    'cc_business_struggles_financial_hardship', 'cc_questionnaire_completion',
    'cc_chasing_response', 'cc_login_issues', 'cc_platform_issues',
    'cc_dissatisfaction_time_to_complete', 'cc_process_complexity_concerns',
    'cc_questions_harder_than_expected', 'cc_dissatisfaction_support',
    'cc_pricing_mentioned', 'cc_pricing_sentiment_impact',
    'cc_refund_discussed'
]

valid_flags = ['Yes', 'No', 'Not Discussed', 'Neutral']

for col in flag_cols:
    if col in df_cc.columns:
        df_cc[col] = df_cc[col].astype(str).str.strip()
        df_cc[col] = df_cc[col].replace({'0': 'No'})
        # Anything not in valid flags (stray numbers, free-text) -> 'Yes'
        invalid_mask = ~df_cc[col].isin(valid_flags)
        df_cc.loc[invalid_mask, col] = 'Yes'

In [0]:
for col in flag_cols:
    if col in df_cc.columns:
        print(f"{col}: {df_cc[col].unique()}")

In [0]:
print("Before cleaning:")
print(df_cc['cc_issues_within_questionnaire'].value_counts())

In [0]:
df_cc['cc_issues_within_questionnaire'] = df_cc['cc_issues_within_questionnaire'].astype(str).str.strip()

# Use pattern matching to catch ALL 'Not applicable...', 'Not mentioned...', etc. variants
lower_vals = df_cc['cc_issues_within_questionnaire'].str.lower()

no_pattern_mask = (
    lower_vals.str.startswith('not applicable') |
    lower_vals.str.startswith('not mentioned') |
    lower_vals.str.startswith('not required') |
    lower_vals.str.startswith('not enough') |
    lower_vals.str.startswith('n/a') |
    (df_cc['cc_issues_within_questionnaire'] == '[Yes/No]') |
    (df_cc['cc_issues_within_questionnaire'] == '0') |
    (df_cc['cc_issues_within_questionnaire'] == 'Not Discussed')
)

# Apply: pattern matches -> 'No'
df_cc.loc[no_pattern_mask, 'cc_issues_within_questionnaire'] = 'No'

# Anything else that is not exactly 'Yes' or 'No' is free-text issue description -> 'Yes'
valid_issue_flags = ['Yes', 'No']
invalid_mask = ~df_cc['cc_issues_within_questionnaire'].isin(valid_issue_flags)
df_cc.loc[invalid_mask, 'cc_issues_within_questionnaire'] = 'Yes'

print("\nAfter cleaning:")
print(df_cc['cc_issues_within_questionnaire'].value_counts())

In [0]:
print("Before cleaning:")
print(df_cc['cc_call_initiated_by'].value_counts())

In [0]:
df_cc['cc_call_initiated_by'] = df_cc['cc_call_initiated_by'].replace({
    'Not Relevant': 'Not Discussed',
    'Not Applicable': 'Not Discussed',
    'No': 'Not Discussed'
})

# Handle [Agent/Customer/Not Relevant] placeholder
df_cc['cc_call_initiated_by'] = df_cc['cc_call_initiated_by'].apply(
    lambda x: 'Not Discussed' if x not in ['Customer', 'Agent', 'Not Discussed'] else x
)

print("After cleaning:")
print(df_cc['cc_call_initiated_by'].value_counts())

In [0]:
print("Before cleaning:")
print(df_cc['cc_care_package'].value_counts())

In [0]:
def normalize_care_package(val):
    if pd.isna(val):
        return 'Not Discussed'
    val_clean = val.strip()
    val_lower = val_clean.lower()
    
    # Not Discussed variants
    if val_lower in ['not discussed', 'nan']:
        return 'Not Discussed'
    if val_lower.startswith('not discussed'):
        return 'Not Discussed'
    if val_lower.startswith('['):
        return 'Not Discussed'
    
    # Standard - includes typos: Standards, Stanford, Slander, Stamp, Sender, 
    # Off-Stander, Start-up, etc.
    if val_clean == 'Standard':
        return 'Standard'
    if val_lower in ['standards', 'stanford', 'slander', 'stamp', 'stamp package',
                     'sender', 'off-stander', 'start-up package', 'class',
                     'sister package', 'rest', 'address', 'sam', 'clickers package',
                     'print', 'press package', 'thunder']:
        return 'Standard'
    
    # Premier - includes Premium, Primary
    if val_clean == 'Premier':
        return 'Premier'
    if val_lower in ['premium', 'primary']:
        return 'Premier'
    
    # Express - includes Fast Track, Fastest, Faster, Fast, Excess, FAFSA, FANDA
    if val_clean == 'Express':
        return 'Express'
    if val_lower in ['fast track', 'fastest', 'faster package', 'fastest package',
                     'fast', 'excess', 'fafsa', 'fanda']:
        return 'Express'
    
    # Assisted - includes Assist, Assistive, Assistance, Assisted Package, etc.
    if val_clean == 'Assisted':
        return 'Assisted'
    if val_lower in ['assist', 'assistive', 'assistance', 'assisted package',
                     'assisted membership', 'assisted audit', 'unassisted']:
        return 'Assisted'
    
    # Combo values - take the first mentioned tier
    if 'standard' in val_lower and 'premier' in val_lower:
        return 'Standard'
    if 'assisted' in val_lower and 'standard' in val_lower:
        return 'Assisted'
    if 'premier' in val_lower and 'express' in val_lower:
        return 'Premier'
    if 'express' in val_lower and 'standard' in val_lower:
        return 'Express'
    if 'funder' in val_lower or 'express' in val_lower:
        return 'Express'
    
    # Catch-all for remaining sponsored/family/gold/other edge cases
    if val_lower in ['sponsored membership', 'family', 'family package', 'gold',
                     '20 working day package']:
        return 'Standard'
    
    # Anything else unrecognizable -> Not Discussed
    return 'Not Discussed'

df_cc['cc_care_package'] = df_cc['cc_care_package'].apply(normalize_care_package)

print("\nAfter cleaning:")
print(df_cc['cc_care_package'].value_counts())

In [0]:
print("Before cleaning:")
print(df_cc['cc_contractor_sentiment'].value_counts())

In [0]:
# Free-text agent comments should map to 'Neutral' since we can't determine sentiment from them
valid_sentiments = ['Dissatisfied', 'Neutral', 'Satisfied', 'Not Discussed']
df_cc['cc_contractor_sentiment'] = df_cc['cc_contractor_sentiment'].astype(str).str.strip()
df_cc['cc_contractor_sentiment'] = df_cc['cc_contractor_sentiment'].apply(
    lambda x: x if x in valid_sentiments else 'Neutral'
)

print("After cleaning:")
print(df_cc['cc_contractor_sentiment'].value_counts())

In [0]:
score_cols = [
    'cc_contractor_sentiment_start_score',
    'cc_contractor_sentiment_end_score',
    'cc_contractor_sentiment_overall_score',
    'cc_contractor_sentiment_issues_score'
]

for col in score_cols:
    print(f"\n--- {col} (before) ---")
    print(df_cc[col].value_counts().head(10))

In [0]:
for col in score_cols:
    # Replace 'Not Discussed' and text labels with NaN, then coerce to numeric
    df_cc[col] = df_cc[col].replace('Not Discussed', np.nan)
    df_cc[col] = pd.to_numeric(df_cc[col], errors='coerce')
    
    # Fill NaN with the median of valid scores (neutral fill)
    median_val = df_cc[col].median()
    df_cc[col] = df_cc[col].fillna(median_val)
    print(f"{col}: median={median_val}, dtype={df_cc[col].dtype}")

In [0]:
print("Before cleaning:")
print(df_cc['cc_contractor_suggest_leave'].value_counts().head(10))

In [0]:
df_cc['cc_contractor_suggest_leave'] = df_cc['cc_contractor_suggest_leave'].astype(str).str.strip()

# 'Not Applicable' -> 'No' (they didn't suggest leaving)
df_cc['cc_contractor_suggest_leave'] = df_cc['cc_contractor_suggest_leave'].replace({'Not Applicable': 'No'})

# Anything not in valid flags is free-text = the topic was discussed = 'Yes'
valid_leave_flags = ['Yes', 'No', 'Not Discussed']
invalid_mask = ~df_cc['cc_contractor_suggest_leave'].isin(valid_leave_flags)
df_cc.loc[invalid_mask, 'cc_contractor_suggest_leave'] = 'Yes'

print("After cleaning:")
print(df_cc['cc_contractor_suggest_leave'].value_counts())

In [0]:
print("Before cleaning:")
print(df_cc['cc_contractor_complained'].value_counts().head(10))

In [0]:
df_cc['cc_contractor_complained'] = df_cc['cc_contractor_complained'].astype(str).str.strip()

# 'Not Applicable' -> 'No'
df_cc['cc_contractor_complained'] = df_cc['cc_contractor_complained'].replace({'Not Applicable': 'No'})

# Anything not in valid flags is a free-text complaint description -> 'Yes'
valid_complained_flags = ['Yes', 'No', 'Not Discussed']
invalid_mask = ~df_cc['cc_contractor_complained'].isin(valid_complained_flags)
df_cc.loc[invalid_mask, 'cc_contractor_complained'] = 'Yes'

print("After cleaning:")
print(df_cc['cc_contractor_complained'].value_counts())

In [0]:
print(df_cc['direction'].value_counts())

In [0]:
df_cc['call_date'] = pd.to_datetime(df_cc['call_date'], format='%d-%m-%Y', errors='coerce')
print(df_cc['call_date'].dtype)
print(df_cc['call_date'].isnull().sum(), "null dates after conversion")

In [0]:
print("Remaining null values:")
print(df_cc.isnull().sum())
print(f"\nTotal rows: {len(df_cc)}")
print(f"Total columns: {len(df_cc.columns)}")

In [0]:
df_cc.dtypes

In [0]:
cat_cols = [
    'direction', 'cc_care_package', 'cc_care_package_discussed',
    'cc_urgency_getting_on_site', 'cc_external_consultant',
    'cc_agent_cross_sell_attempt', 'cc_customer_issues_concerns',
    'cc_business_struggles_financial_hardship', 'cc_call_initiated_by',
    'cc_questionnaire_completion', 'cc_chasing_response',
    'cc_issues_within_questionnaire', 'cc_login_issues',
    'cc_platform_issues', 'cc_dissatisfaction_time_to_complete',
    'cc_process_complexity_concerns', 'cc_questions_harder_than_expected',
    'cc_dissatisfaction_support', 'cc_contractor_sentiment',
    'cc_pricing_mentioned', 'cc_pricing_sentiment_impact',
    'cc_refund_discussed', 'cc_contractor_suggest_leave',
    'cc_contractor_complained'
]

for col in cat_cols:
    df_cc[col] = df_cc[col].astype('category')

In [0]:
df_cc.info()


In [0]:
# Use df_cc instead of spark.table("post_renewal_churn.raw.renewal_calls")
df_calls = df_cc[df_cc['co_ref'].notnull()]

# Convert pandas DataFrame to Spark DataFrame
df_calls_spark = spark.createDataFrame(df_calls)
df_calls_spark.write.mode("overwrite").saveAsTable("workspace.`ds-datasets`.cc_calls")


In [0]:
df_calls_spark.select("co_ref").count()

In [0]:
import pyspark.sql.functions as F
from pyspark.sql.window import Window # ── 1. Load / prepare billings ──────────────────────────────────────────


df_billings = spark.table("workspace.`ds-raw-datasets`.raw_billings") \
    .filter(F.col("prospect_outcome") != "Open") \
    .withColumn("datediff", F.datediff("closed_date", "prospect_renewal_date")) \
    .filter(F.col("datediff") < 29) \
    .withColumn("index", F.monotonically_increasing_id())

# Use df_cc instead of spark.table("post_renewal_churn.raw.renewal_calls")
df_calls = df_cc[df_cc['co_ref'].notnull()]

# Convert pandas DataFrame to Spark DataFrame
df_calls_spark = spark.createDataFrame(df_calls)

# ── 3. Join on co_ref — explicit condition to avoid ambiguous reference ──
df_joined = df_billings.join(
    df_calls_spark,
    on=df_billings["co_ref"] == df_calls_spark["co_ref"],
    how="left"
).filter(
    (F.col("Call_Date") >= F.col("prospect_renewal_date"))
).drop(df_calls_spark["co_ref"])

# ── 4. Keep only the LATEST call per billing row (index) ────────────────
window = Window.partitionBy("index").orderBy(F.desc("Call_Date"))

df_latest_call = df_joined \
    .withColumn("rn", F.row_number().over(window)) \
    .filter(F.col("rn") == 1) \
    .drop("rn")

display(df_latest_call)
df_final = df_latest_call


In [0]:
c=df_final.columns

In [0]:
from pyspark.sql import functions as F
from scipy.stats import chi2_contingency, ttest_ind
import pandas as pd

# Columns to test (from your schema)
columns = [
    "Contact_ID", "Call_Date", "Direction", "cc_care_package", "cc_care_package_discussed",
    "cc_urgency_getting_on_site", "cc_external_consultant", "cc_agent_cross_sell_attempt",
    "cc_customer_issues_concerns", "cc_business_struggles_financial_hardship", "cc_call_initiated_by",
    "cc_questionnaire_completion", "cc_chasing_response", "cc_issues_within_questionnaire",
    "cc_login_issues", "cc_platform_issues", "cc_dissatisfaction_time_to_complete",
    "cc_process_complexity_concerns", "cc_questions_harder_than_expected", "cc_dissatisfaction_support",
    "cc_contractor_sentiment", "cc_contractor_sentiment_start_score", "cc_contractor_sentiment_end_score",
    "cc_contractor_sentiment_overall_score", "cc_contractor_sentiment_issues_score", "cc_pricing_mentioned",
    "cc_pricing_sentiment_impact", "cc_refund_discussed", "cc_contractor_suggest_leave",
    "cc_contractor_complained", "Co_Ref", "Analysed_Call", "Call_Year"
]

# Categorical columns for chi-square test
categorical_cols = [
    "Direction", "cc_care_package", "cc_care_package_discussed", "cc_urgency_getting_on_site",
    "cc_external_consultant", "cc_agent_cross_sell_attempt", "cc_customer_issues_concerns",
    "cc_business_struggles_financial_hardship", "cc_call_initiated_by", "cc_questionnaire_completion",
    "cc_chasing_response", "cc_issues_within_questionnaire", "cc_login_issues", "cc_platform_issues",
    "cc_dissatisfaction_time_to_complete", "cc_process_complexity_concerns", "cc_questions_harder_than_expected",
    "cc_dissatisfaction_support", "cc_contractor_sentiment", "cc_pricing_mentioned",
    "cc_pricing_sentiment_impact", "cc_refund_discussed", "cc_contractor_suggest_leave",
    "cc_contractor_complained", "Analysed_Call", "Call_Year"
]

# Numeric columns for t-test
numeric_cols = [
    "Contact_ID", "cc_contractor_sentiment_start_score", "cc_contractor_sentiment_end_score",
    "cc_contractor_sentiment_overall_score", "cc_contractor_sentiment_issues_score"
]

# Prepare DataFrame for testing
df_test = df_final.select(columns + ["prospect_outcome"])

results = []

for col in columns:
    h0 = f"There is no association between {col} and prospect_outcome."
    h1 = f"There is an association between {col} and prospect_outcome."
    if col in categorical_cols:
        pdf = df_test.groupBy(col, "prospect_outcome").count().toPandas().pivot(index=col, columns="prospect_outcome", values="count").fillna(0)
        chi2, p, _, _ = chi2_contingency(pdf.values)
        result = "Reject H0" if p < 0.05 else "Fail to reject H0"
        results.append((col, h0, h1, p, result))
    elif col in numeric_cols:
        pdf = df_test.select(col, "prospect_outcome").toPandas()
        groups = pdf.groupby("prospect_outcome")[col]
        if len(groups) == 2:
            vals1 = groups.get_group(list(groups.groups.keys())[0]).dropna()
            vals2 = groups.get_group(list(groups.groups.keys())[1]).dropna()
            t_stat, p = ttest_ind(vals1, vals2)
            result = "Reject H0" if p < 0.05 else "Fail to reject H0"
            results.append((col, h0, h1, p, result))
        else:
            results.append((col, h0, h1, None, "Not enough groups for t-test"))
    else:
        results.append((col, h0, h1, None, "Not tested"))

for col, h0, h1, p, result in results:
    print(f"Column: {col}")
    print(f"H0: {h0}")
    print(f"H1: {h1}")
    print(f"p-value: {p}")
    print(f"Result: {result}\n")

In [0]:
reject_ho_columns = [
    "Co_Ref",
    "Call_Date",
    "Contact_ID",
    "Direction",
    "cc_business_struggles_financial_hardship",
    "cc_platform_issues",
    "cc_process_complexity_concerns",
    "cc_contractor_sentiment_start_score",
    "cc_contractor_sentiment_issues_score",
    "cc_pricing_sentiment_impact",
    "cc_contractor_complained",
    "Analysed_Call"
]

df_calls_spark.select(*reject_ho_columns).write.mode("overwrite").saveAsTable("workspace.`ds-raw-datasets`.cc_calls_cleaned_final")